# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
dbutils.widgets.text("ruta_csv", "file:/Workspace/FilesLab2p2/ventas_export.csv")
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

filas_oro_roto
59


## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# 1. Crear vista intermedia evaluando reglas y asignando número de fila por duplicado
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_evaluadas AS
SELECT 
    *,
    TRY_CAST(tienda_id AS INT) AS tienda_limpia,
    TRY_CAST(REPLACE(monto, ',', '.') AS DOUBLE) AS monto_limpio,
    ROW_NUMBER() OVER(PARTITION BY venta_id ORDER BY venta_id) AS rn
FROM bronce_ventas
""")

# 2. Crear tabla de hechos limpia (oro_fact_ventas)
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT 
    venta_id, 
    fecha, 
    tienda_limpia AS tienda_id, 
    producto_id, 
    cantidad, 
    monto_limpio AS monto, 
    medio_pago
FROM ventas_evaluadas
WHERE rn = 1 
  AND tienda_limpia IN (1, 2, 3) 
  AND monto_limpio IS NOT NULL
""")

# 3. Crear tabla de cuarentena con motivo (cuarentena_ventas)
spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT 
    venta_id, 
    fecha, 
    tienda_id, 
    producto_id, 
    cantidad, 
    monto, 
    medio_pago, 
    canal,
    CASE 
        WHEN rn > 1 THEN 'Venta duplicada'
        WHEN tienda_limpia NOT IN (1, 2, 3) OR tienda_limpia IS NULL THEN 'Tienda no autorizada'
        WHEN monto_limpio IS NULL THEN 'Monto inválido o corrupto'
    END AS motivo
FROM ventas_evaluadas
WHERE rn > 1 
   OR tienda_limpia NOT IN (1, 2, 3) OR tienda_limpia IS NULL 
   OR monto_limpio IS NULL
""")

DataFrame[]

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
r/ Por medio de una función ROW_NUMBER se agrupan y numeran los registros segun su id de venta, para el caso de la primera aparicion, esta se queda guardada en la tabla de hechos de oro, con ello ya tenemos una especie de "banderín" que nos va indicar que el unico cupo disponible ya fue utilizado por tanto, el duplicado, triplicado o el X ado que se haga de la venta sera enviado automaticamente hacia el apartado de cuarentena, esto con el justificante o bajo el proceso de descarte logistico de ser un dato duplicado/repetido.
2. ¿Qué hicieron con un monto que venía en texto?
r/ Lo primordial fue realizar una limpieza del dato especificamente su formato por medio de REPLACE, para el caso especifico fueron comas por puntos, adicionalmente se hizo uso de TRY_CAST con el fin de "forzar" que el sistema trabaje bajo un solo formato para el caso del laboratorio fue un DOUBLE, entonces, lo que sucedia era que, si el texto era ilegible se asumiria un valor NULL con ello la forzamos a un monto invalido para que luego proceda a trasladarse ese dato hacia la cuarentena.
3. ¿Por qué una tienda se quedó fuera?
r/ Las reglas del negocio nos limitan a un máximo admisible de 3 tiendas, estas ya definidas en el pipeline, por ende, si quisieramos hacer un registro de alguna otra tienda o en su defecto algun registro que por algún motivo viniese vacio, este comportamiento o solicitud se asumira como un incumplimiento a la regla y por consecuente ser trasladado hacia la cuarentena.
4. ¿Dónde quedó la columna que no estaba en el contrato?
r/Por motivos contractuales esta columna (canal), no fue incluida dentro del SELECT, pues el contrato nos da un esquema no modificable a seguir y por motivos legales contractuales esto debe respetarse, pero, siempre abogando por la legal y etica, para evitar caer en otro problema de esta indole si se incluyo una tabla de cuarentena de ventas para asi mantaner una visibilidad de los datos originales en registros ajenos a lo acordado en el esquema contractual.
5. Si corren el notebook otra vez, ¿el hecho crece?
r/ La respuesta corta es no, pues, con un CREATE OR REPLACE TEMP VIEW lo que hacemos es borrar las vistas anteriores y reconstruir una nueva desde cero, es decir desde la fase de raw data, el motivo de esto es abogar por una garantia de idempotencia.

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio